In [13]:
import json
import polars as pl
import duckdb
from xlsxwriter import Workbook
from openpyxl import load_workbook

In [14]:
# get motherduck token
server_config = "/home/asha/airflow/duckdb-config.json"

with open(server_config, "r") as fp:
    config = json.load(fp)
token = config['token']

con = duckdb.connect(f'md:?motherduck_token={token}')
con.sql("use asha_production;")

In [15]:
schemas = ['bronze', 'silver', 'gold']
objects = []
for schema in schemas:
    temp_df = con.sql(f"show tables from {schema};").pl()
    temp_df = temp_df.with_columns(
            pl.lit(schema).alias('schema')
        )
    objects.append(temp_df)

objectDF = pl.concat(objects)

In [16]:
schemas = ['bronze', 'silver', 'gold']
objectColumns = []
for schema in schemas:
    
    df = con.sql(f"show tables from {schema};").pl()
    tableList = df['name'].to_list()

    for tbl in tableList:
        temp_df = con.sql(f"describe {schema}.{tbl}").pl()
        temp_df = temp_df.with_columns(
            pl.lit(schema).alias('schema'),
            pl.lit(tbl).alias('table_name')
        )
        objectColumns.append(temp_df)

objectColumnsDF = pl.concat(objectColumns)

In [17]:
schemas = ['bronze', 'silver', 'gold']
objectPragmaColumns = []
for schema in schemas:
    
    df = con.sql(f"show tables from {schema};").pl()
    tableList = df['name'].to_list()

    for tbl in tableList:
        temp_df = con.sql(f"pragma table_info('{schema}.{tbl}')").pl()
        temp_df = temp_df.with_columns(
            pl.lit(schema).alias('schema'),
            pl.lit(tbl).alias('table_name')
        )
        objectPragmaColumns.append(temp_df)

objectPragmaColumnsDF = pl.concat(objectPragmaColumns)

In [18]:
with Workbook("data_governance.xlsx") as wb:  
    # basic/default conditional formatting
    objectDF.write_excel(
        workbook=wb,
        worksheet="objects",
        position=(1,0),  # specify position as (row,col) coordinates
        table_style="Table Style Light 1",
        autofit=True
    )
    
    ws = wb.get_worksheet_by_name('objects')
    fmt_title = wb.add_format(
        {
            'font_color': '#000000',
            'font_size': 16,
            'italic': False,
            'bold': True
            
        }
    )
    
    ws.write(0, 0, "Objects", fmt_title)
    
    objectColumnsDF.write_excel(
        workbook=wb,
        worksheet="object_columns",
        position=(0,0),  # specify position as (row,col) coordinates
        table_style="Table Style Light 1",
        autofit=True
    ) 
    objectPragmaColumnsDF.write_excel(
        workbook=wb,
        worksheet="object_columns_pragma",
        position=(0,0),  # specify position as (row,col) coordinates
        table_style="Table Style Light 1",
        autofit=True
    )